In [ ]:
fecha_mes_base='2026-08-01'
tipi_cond1='SAE'
tipi_cond2='SAR'
tipi_cond3='AGENDA'
tb_tipolofia='tTipologia_Cencosud_PPFF'
servidor_01=76
tipi_cod='Codigo'
tipi_resp_cod='R0'
tipi_descrip='DESCRIPCION'
tipi_estado='tipo'
tipi_resp_estado='NO GESTIONADO'
tipi_subdescripcion='SUB_DESCRIPCION'
tnum_tb='tNumeroCenco_Sae'
tnum_dni='CODDOC'
tlista_generada='borrar_prestamo_cencosud'
get_base=since_base_maestra_cencosud_ppff

def resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado):
    query = f"""
        SELECT *
        FROM OPENQUERY([192.168.3.{servidor_01}], '
            SELECT        
            rtrim(ltrim(d.vendor_lead_code)) AS vendor_lead_code,        
            e.dial_method,
            a.campaign_id AS numero_campana,        
            a.user AS dni_ejecutivo,
            c.full_name AS ejecutivo,
            e.campaign_name AS nombre_campana,        
            a.call_date AS fecha_hora_llamada,        
            a.length_in_sec AS duracion,        
            b.status_name AS call_result,        
            f.list_description,        
            f.list_name,        
            a.phone_number as phone_number,        
            d.alt_phone as fecha_agenda,        
            d.comments as comentarios,        
            a.status AS codigo,
            a.term_reason,	
            a.alt_dial
            FROM asterisk.vicidial_log a         
            LEFT JOIN asterisk.vicidial_list d ON a.lead_id=d.lead_id        
            LEFT JOIN asterisk.vicidial_campaigns e ON a.campaign_id=e.campaign_id        
            LEFT JOIN asterisk.vicidial_lists f ON a.list_id=f.list_id        
            LEFT JOIN asterisk.vicidial_statuses b ON a.status=b.status        
            LEFT JOIN asterisk.vicidial_users c ON a.user=c.user        
            WHERE (e.campaign_name like "%{tipi_cond1}" or e.campaign_name like "%{tipi_cond2}" or e.campaign_name like "%{tipi_cond3}")
            AND a.call_date >= DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01'')
            AND a.call_date < 
            DATE_ADD(DATE_FORMAT(''{fecha_mes_base}'', ''%Y-%m-01''), INTERVAL 1 MONTH)
        ')

        """
    df_vicidial=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial = df_vicidial.withColumn(
        "vendor_lead_code",
        F.lpad(F.col("vendor_lead_code").cast("string"), 8, "0")
    )
    query = f"""
        SELECT {tipi_cod} as codigo
        , case
            when {tipi_cod}='CALLBK' then 'VOLVER A LLAMAR - call'
            else {tipi_descrip} 
        end as descripcion
        ,case 
            when {tipi_cod}='CALLBK' then 1200
            else peso 
        end as peso  FROM [ODIN].[dbo].{tb_tipolofia}
        where LEFT({tipi_cod},2)='{tipi_resp_cod}' or {tipi_estado}='{tipi_resp_estado}' or {tipi_cod}='CALLBK'
        """
    df_tipi=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

    df_vicidial=df_vicidial.join(df_tipi,["codigo"],"left")

    return df_vicidial.select('fecha_hora_llamada','list_name','vendor_lead_code','phone_number','descripcion','dni_ejecutivo','ejecutivo','dial_method','term_reason','alt_dial','call_result','duracion','codigo','nombre_campana')

df_vici=resumen_vicidial(spark,fecha_mes_base,tipi_cond1,tipi_cond2,tipi_cond3,tb_tipolofia,servidor_01,tipi_cod,tipi_resp_cod,tipi_descrip,tipi_estado,tipi_resp_estado)



In [10]:
df_agendas=df_vici.filter(F.col('dni_ejecutivo').isin('PFC120','PFC113')).orderBy(F.col('fecha_hora_llamada').asc_nulls_last())

In [12]:
print(df_agendas.columns)

['fecha_hora_llamada', 'list_name', 'vendor_lead_code', 'phone_number', 'descripcion', 'dni_ejecutivo', 'ejecutivo', 'dial_method', 'term_reason', 'alt_dial', 'call_result', 'duracion', 'codigo', 'nombre_campana']


In [11]:
print([row['descripcion' ] for row in df_agendas.select('descripcion').distinct().collect()])


['VOLVER A LLAMAR', 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA', 'NO UTILIZA (TARJETAS - PRESTAMOS)', 'OFERTA DE TASA MUY ALTA', 'OFERTA DE LINEA MUY BAJA', 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)', 'VOLVER A LLAMAR (TERCERO RELACIONADO)', 'NO VOLVER A LLAMAR NUNCA MAS', 'TELEFONO OCUPADO / NO CONTESTAN', 'NO DESEA –NO ESPECIFICA MOTIVO', 'TELEFONO EQUIVOCADO', 'GESTION EN PROCESO (AUTO)', 'CLIENTE FALLECIO', 'CLIENTE ACEPTA PRODUCTO', 'TELEFONO FUERA DE SERVICIO / NO EXISTE']


In [14]:
dni_volver = (
    df_agendas
    .select('vendor_lead_code')
    .filter(F.upper(F.col("descripcion")) == "VOLVER A LLAMAR")
    .select("vendor_lead_code")
    .distinct()
)

df_agendas_01 = df_agendas.join(dni_volver,["vendor_lead_code"],"inner")

In [16]:
from pyspark.sql import Window
from pyspark.sql import functions as F

w = Window.partitionBy('dni_ejecutivo',"vendor_lead_code") \
          .orderBy(F.col("fecha_hora_llamada").asc())

df_ultima = df_agendas_01.withColumn("rn", F.row_number().over(w))


In [18]:
df_ultima.filter(
    (F.col('vendor_lead_code')=='02432060')&
    (F.col('dni_ejecutivo')=='PFC120')
).show()

+----------------+-------------------+--------------------+------------+--------------------+-------------+--------------------+--------------------+------------+--------+-----------+--------+------+--------------+---+
|vendor_lead_code| fecha_hora_llamada|           list_name|phone_number|         descripcion|dni_ejecutivo|           ejecutivo|         dial_method| term_reason|alt_dial|call_result|duracion|codigo|nombre_campana| rn|
+----------------+-------------------+--------------------+------------+--------------------+-------------+--------------------+--------------------+------------+--------+-----------+--------+------+--------------+---+
|        02432060|2026-08-01 11:56:09|montosaltostasasb...|   999878111|     VOLVER A LLAMAR|       PFC120|GIOVANNA CORTEZ M...|RATIO            ...|CALLER      |    MAIN|       NULL|      58|  R003|   2026-08 SAE|  1|
|        02432060|2026-08-03 09:33:25|montosaltostasasb...|   999878111|     VOLVER A LLAMAR|       PFC120|GIOVANNA CORTEZ M

In [19]:
df_ultima=df_ultima.withColumn('agendas',when(F.col('rn')==1,1).otherwise(F.lit(0)))

In [21]:
df_ultima_pd=df_ultima.toPandas()

In [23]:
ruta_archivo = os.path.join(ruta_csv, 'agndas_cenco.csv')
df_ultima_pd.to_csv(ruta_archivo, index=False,sep=';')

In [ ]:
['VOLVER A LLAMAR', 'CLIENTE CUELGA ANTES DE ESCUCHAR OFERTA', 'NO UTILIZA (TARJETAS - PRESTAMOS)', 'OFERTA DE TASA MUY ALTA', 'OFERTA DE LINEA MUY BAJA', 'NO SE CONCRETO LLAMADA (CASILLA DE VOZ - APAGADO)', 'VOLVER A LLAMAR (TERCERO RELACIONADO)', 'NO VOLVER A LLAMAR NUNCA MAS', 'TELEFONO OCUPADO / NO CONTESTAN', 'NO DESEA –NO ESPECIFICA MOTIVO', 'TELEFONO EQUIVOCADO', 'GESTION EN PROCESO (AUTO)', 'CLIENTE FALLECIO', 'CLIENTE ACEPTA PRODUCTO', 'TELEFONO FUERA DE SERVICIO / NO EXISTE']call


In [ ]:
['PFC113', 'PFC114', 'TCD072', 'PFC118', 'PFC003', 'PFC007', 'PFC119', 'PFC111', '40940689', 'PFC092', 'VDAD', 'PFC120', 'PFC021', 'PFC112', 'PFC005', 'PFC008']113

In [ ]:
df_vici.filter(F.col('vendor_lead_code')=='45642470').orderBy(F.col('fecha_hora_llamada').desc()).show(truncate=False)


### simular

In [3]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [2]:

filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')

filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())



C:\Users\DATA\AppData\Local\Temp\ipykernel_22840\2273191912.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [ ]:
query = """
    select NUMERO_DOCUMENTO as Dni,cl_telf1 as PHONE_NUMBER
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente
    where cl_base='julio 2026'
    and cl_telf1<>0
    and cl_telf1 is not null
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)







['Dni', 'PHONE_NUMBER']


In [37]:
query = """
    select NUMERO_DOCUMENTO as Dni,PROPENSION_IC,FRESCURA,USER_V3,TIPO_LOTE,lote,recorrido
    from maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    where retiro='Activo'

    """
df_ref_1=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_ref_1=df_ref_1.join(df_formato,['Dni'],'inner')
print(df_ref_1.count())

173214


In [23]:
df_ref_1.show(3)

+--------+-------------+--------+--------------------+------------+----------+---------+------------+
|     Dni|PROPENSION_IC|FRESCURA|             USER_V3|   TIPO_LOTE|      lote|recorrido|PHONE_NUMBER|
+--------+-------------+--------+--------------------+------------+----------+---------+------------+
|00367836|            3|       4|14. Otros Bancari...|    BASE BOT|       BOT|        1|   972692631|
|02165933|            2|       4|            6. MES B|BASE REGULAR|NO CLIENTE|        1|   906093531|
|02615064|            1|       4|            7. Peers|BASE REGULAR|NO CLIENTE|        1|   939686257|
+--------+-------------+--------+--------------------+------------+----------+---------+------------+
only showing top 3 rows


In [20]:



filename='subir_001.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_subir_01 = cargar_archivo_csv(spark,filename,';',True)


In [27]:
df_ref_1=df_ref_1.join(df_todo,['Dni'],'inner')

In [38]:
df_todo=df_todo.withColumnRenamed('vendor_lead_code','Dni')

In [34]:
df_es=cargar_archivo_csv(spark,'queda_alfin.csv',';',True)
df_todo=cargar_archivo_csv(spark,'alfin_RE.csv',';',True)
df_todo=df_todo.select('vendor_lead_code')
print(df_es.columns)
print(df_todo.columns)

['_c0', '0']
['vendor_lead_code']


In [ ]:
=df_todo.select('Dni')

In [ ]:
df_subir_01

In [21]:


df_subir_01 = df_subir_01.withColumn(
    "NUMERO_DOCUMENTO",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )

)
df_subir_01=df_subir_01.filter(F.col('NUMERO_DOCUMENTO').isNotNull())

In [7]:
df_ref=df_ref_1.filter(
        # ((F.col('lote')=='NO CLIENTE')&(F.col('PROPENSION_IC').isin(3))&(F.col('recorrido')==0))|
        (
           ( F.col('lote')=='BOT')&(F.col('PROPENSION_IC').isin(3,4))
        )
    )
df_ref.count()

13744

In [ ]:
df_ref_1

In [ ]:
df_retiro1=cargar_archivo_csv()

In [42]:
query = """
SELECT *
FROM OPENQUERY([192.168.2.100], '
    select *,dni_cliente as Dni from Alice.prospectos_correos_alfin
    WHERE fecha_dia IN (''2026-07-15'', ''2026-07-15'')
')
    """
df_crm=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_crm=df_crm.select( 'Dni', 'fecha_visita', 'hora_visita', 'fecha_envio', 'intentos_realizados', 'fecha_registro','estado')

df_crm=df_crm.withColumn('ref',when(F.col('estado')=='ENVIADO',1).otherwise(2))

window_spec = Window.partitionBy("Dni").orderBy(col("ref").asc_nulls_last())
df_crm = df_crm.withColumn("n_mejor_resul_dia", row_number().over(window_spec))
df_crm=df_crm.filter((F.col('n_mejor_resul_dia')==1)&(F.col('ref')==1)).drop('ref','n_mejor_resul_dia')

In [43]:
# df_crm=df_crm.withColumnRenamed('Dni','DNI')
df_crm = df_crm.withColumn(
    "Dni",
    F.right(
        F.concat(F.lit("00000000"), F.col("Dni")),
        F.lit(8)
    )

)


In [44]:
df_ref_f=df_ref.join(df_crm,['Dni'],'left')

In [22]:
df_subir_01=df_subir_01.withColumnRenamed('dni_cliente','Dni')

In [40]:
w = Window.partitionBy('lote','PROPENSION_IC','FRESCURA','USER_V3').orderBy(F.rand())

df_split = df_ref_f.withColumn("por_dia", F.ntile(1).over(w))

In [34]:
df_split=df_split.withColumn('por_dia',when(F.col('por_dia')==2,1).otherwise(F.col('por_dia')))

In [28]:

df_split = df_split.withColumn(
    "por_dia",
    F.when(F.to_date("fecha_registro") == F.lit("2026-07-08"), 6)
     .when(F.to_date("fecha_registro") == F.lit("2026-07-09"), 7)
     .otherwise(F.col("por_dia"))
)

In [23]:
df_subir_01=df_subir_01.withColumn('fecha_registro',F.lit('2026-07-27'))


In [25]:
df_split=df_subir_01.withColumn('Numero_Campana',F.lit('401'))
df_split=df_split.withColumn('list_description',F.lit('provicional'))
df_split=df_split.withColumn('list_name',F.lit('provicional'))
# df_split=df_split.withColumn('Nombre_Campana',when(F.lit('lote')=='BOT',F.lit('BOT_ALFIN')).otherwise(F.lit('2026-07 BCO ALFIN')))


In [27]:
df_split=df_split.withColumn('Nombre_Campana',F.lit('2026-07 BCO ALFIN'))
df_split=df_split.withColumn('Fecha_Llamada',F.lit('2026-07-27'))

In [12]:
df_split=df_split.withColumn('por_dia',when(F.col('por_dia')==5,6)
.when(F.col('por_dia')==6,7)
.when(F.col('por_dia')==7,8).otherwise(F.col('por_dia')))

In [42]:
df_split = df_split.withColumn(
    "Fecha_Llamada",
    F.to_date(
        F.concat(
            F.lit("2026-07-"),
            F.lpad(F.col("por_dia").cast("string"), 2, "0")
        )
    )
)

In [14]:

query = """
select top(10)* From SAMANTHA.dbo.tmp_llamadas_mes
    """
df_necesito=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_necesito=df_necesito.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description')

In [49]:
query = """
select top(10)* From SAMANTHA.dbo.tmp_llamadas_mes
    """
df_necesito=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_necesito.show(4)

+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|     Dni|Numero_Campana|      Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|Descripcion|    list_description|           list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin|lead_id|
+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|72048296|            80|2023-11 CENCOSUD ...|         VDAD|Outbound Auto Dial|

In [28]:
# usuarios_regular = [
#     "PEC200",
#     "PEC188",
#     "PEC136",
#     "PEC139",
#     "PEC184",
#     'VDAD'
# ]
usuarios_bot = [
    "49587612",
    'VDAD'
]

In [ ]:
usuarios_bot = [
    "49587612",
    'VDAD'
]

In [29]:
df_split=df_split.withColumn('TIPO_LOTE',F.lit('BASE BOT'))


In [ ]:
# df_crm=df_crm.withColumnRenamed('Dni','DNI')


In [ ]:
df_ref=df_ref.select('PROPENSION_IC','DNI','FRESCURA')
df_ref=df_ref.withColumnRenamed('DNI','Dni')
df_ref = df_ref.withColumn(
    "Dni",
    F.right(
        F.concat(F.lit("00000000"), F.col("Dni")),
        F.lit(8)
    )

)

# df_ref_pd=df_ref.toPandas()

In [35]:
df_split.head()

Row(_c0='0', Dni='10023403', celular='922098386', PESO1='2.0', retiro='0', NUMERO_DOCUMENTO='10023403', fecha_registro='2026-07-27', Numero_Campana='401', list_description='provicional', list_name='provicional', Nombre_Campana='2026-07 BCO ALFIN', Fecha_Llamada='2026-07-27', TIPO_LOTE='BASE BOT')

In [36]:
df_split=df_split.join(df_ref,['Dni'],'left')

In [ ]:
filename='BASE_TARGET_20260727.csv'
df_ref=cargar_archivo_csv(spark,filename,';',True)
print(df_ref.columns)
['PROPENSION_IC', 'USER_V3', 'DNI', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'CampaÃ±a', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÃ‘A_ANTERIOR', 'VARIACION_TASA_CAMPAÃ‘A_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP']fres

['PROPENSION_IC', 'USER_V3', 'DNI', 'X_APPATERNO', 'X_APMATERNO', 'X_NOMBRE', 'OFERTA_MAX', 'tasa_minima', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'TIPO_GEST', 'TIPO_CLIENTE_COMERCIAL', 'CampaÃ±a', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'PERFIL_RO', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SUCURSAL_COMERCIAL', 'Agencia_comercial', 'REGION_COMERCIAL', 'VARIACION_OFERTA_CAMPAÃ‘A_ANTERIOR', 'VARIACION_TASA_CAMPAÃ‘A_ANTERIOR', 'VAR_TASA_CREDITO_ANTERIOR', 'FLG_CET_6M', 'FLAG_DEUDA_V_OFERTA', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'BLOQUE', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'CAMP_BONO', 'CAMP_ADP']


In [39]:

# Probabilidad base de VDAD por TIPO_LOTE
df_split = df_split.withColumn(
    "prob_vdad_base",
    F.when(F.col("TIPO_LOTE") == "BASE REGULAR", F.lit(0.43))
     .when(F.col("TIPO_LOTE") == "BASE BOT", F.lit(0.92))
     .otherwise(F.lit(0.43))
)

# Score: mientras más alto PROPENSION_IC y FRESCURA, más chance de VDAD
df_split = df_split.withColumn(
    "score_vdad",
    (
        (F.col("PROPENSION_IC") - 1) / F.lit(5) +   # PROPENSION_IC 1 a 6
        (F.col("FRESCURA") / F.lit(5))           # FRESCURA 0 a 5
    ) / F.lit(2)
)

# Ajustamos la probabilidad de VDAD
# Ejemplo: puede subir hasta +15 puntos porcentuales
df_split = df_split.withColumn(
    "prob_vdad_final",
    F.least(
        F.col("prob_vdad_base") + (F.col("score_vdad") * F.lit(0.15)),
        F.lit(0.99)
    )
)

# Número aleatorio
df_split = df_split.withColumn("rnd", F.rand())

# Asignar primero VDAD según probabilidad
df_split = df_split.withColumn(
    "DNI_Ejecutivo",
    F.when(F.col("rnd") <= F.col("prob_vdad_final"), F.lit("VDAD"))
)

# Para los demás, asignación uniforme según TIPO_LOTE
# regular_sin_vdad = [x for x in usuarios_regular if x != "VDAD"]
bot_sin_vdad = [x for x in usuarios_bot if x != "VDAD"]

df_split = df_split.withColumn(
    "DNI_Ejecutivo",
    # F.when(
    #     F.col("DNI_Ejecutivo").isNull() & (F.col("TIPO_LOTE") == "BASE REGULAR"),
    #     F.array(*[F.lit(x) for x in regular_sin_vdad])[
    #         F.floor(F.rand() * F.lit(len(regular_sin_vdad))).cast("int")
    #     ]
    # )
    F.when(
        F.col("DNI_Ejecutivo").isNull() & (F.col("TIPO_LOTE") == "BASE BOT"),
        F.array(*[F.lit(x) for x in bot_sin_vdad])[
            F.floor(F.rand() * F.lit(len(bot_sin_vdad))).cast("int")
        ]
    )
    .otherwise(F.col("DNI_Ejecutivo"))
)

df_split = df_split.drop("prob_vdad_base", "score_vdad", "prob_vdad_final", "rnd")

In [40]:
df_split = df_split.withColumn(
    "Ejecutivo",
    F.when(
        (F.col("TIPO_LOTE") == "BASE BOT") & (F.col("DNI_Ejecutivo") == "VDAD"),
        F.lit("Outbound Auto Dial")
    )
    .when(
        (F.col("TIPO_LOTE") == "BASE BOT") & (F.col("DNI_Ejecutivo") != "VDAD"),
        F.lit("Bot Bco Alfin")
    )
    # .when(
    #     (F.col("TIPO_LOTE") == "BASE REGULAR") & 
    #     (F.col("DNI_Ejecutivo").isin([u for u in usuarios_regular if u != "VDAD"])),
    #     F.lit("Agente BCO Alfin")
    # )
    # .when(
    #     (F.col("TIPO_LOTE") == "BASE REGULAR") & (F.col("DNI_Ejecutivo") == "VDAD"),
    #     F.lit("Outbound Auto Dial")
    # )
    .otherwise(F.lit(None))
)

In [41]:
codigos_vdad = ["PDROP", "NA", "AB"]

# codigos_regular = ["zzz5", "zzz6", "zzz9", "zzz13", "zzz10", "zzz11", "zzz56", "zzz23", "zzz24", "zzz26"]

codigos_bot = ["zzz5", "zzz10", "zzz19", "zzz22", "zzz23", "zzz15", "zzz24", "zzz26"]

arr_vdad = F.array(*[F.lit(x) for x in codigos_vdad])
# arr_regular = F.array(*[F.lit(x) for x in codigos_regular])
arr_bot = F.array(*[F.lit(x) for x in codigos_bot])

df_split = df_split.withColumn(
    "Codigo_Paleta",
    F.when(
        F.col("DNI_Ejecutivo") == "VDAD",
        arr_vdad[F.floor(F.rand() * F.lit(len(codigos_vdad))).cast("int")]
    )
    # .when(
    #     (F.col("TIPO_LOTE") == "BASE REGULAR") & (F.col("DNI_Ejecutivo") != "VDAD"),
    #     arr_regular[F.floor(F.rand() * F.lit(len(codigos_regular))).cast("int")]
    # )
    .when(
        (F.col("TIPO_LOTE") == "BASE BOT") & (F.col("DNI_Ejecutivo") != "VDAD"),
        arr_bot[F.floor(F.rand() * F.lit(len(codigos_bot))).cast("int")]
    )
    .otherwise(F.lit(None))
)

In [50]:
df_split=df_split.drop('PESO1')

In [ ]:
XDXD

In [ ]:

filename='subir_001.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_subir_01 = cargar_archivo_csv(spark,filename,';',True)
df_subir_01.show()

In [54]:
df_subir_01.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- dni_cliente: string (nullable = true)
 |-- celular: string (nullable = true)
 |-- PESO1: string (nullable = true)
 |-- retiro: string (nullable = true)



In [55]:
from pyspark.sql import functions as F

df_subir_01 = df_subir_01.withColumn(
    "PESO1",
    F.expr("try_cast(try_cast(trim(PESO1) as double) as int)")
)
df_subir_01.printSchema()


root
 |-- _c0: string (nullable = true)
 |-- dni_cliente: string (nullable = true)
 |-- celular: string (nullable = true)
 |-- PESO1: integer (nullable = true)
 |-- retiro: string (nullable = true)



In [51]:
df_split.show()

+--------+---+---------+------+----------------+--------------+--------------+----------------+-----------+-----------------+-------------+---------+-------------+--------+-------------+------------------+-------------+
|     Dni|_c0|  celular|retiro|NUMERO_DOCUMENTO|fecha_registro|Numero_Campana|list_description|  list_name|   Nombre_Campana|Fecha_Llamada|TIPO_LOTE|PROPENSION_IC|FRESCURA|DNI_Ejecutivo|         Ejecutivo|Codigo_Paleta|
+--------+---+---------+------+----------------+--------------+--------------+----------------+-----------+-----------------+-------------+---------+-------------+--------+-------------+------------------+-------------+
|10023403|  0|922098386|     0|        10023403|    2026-07-27|           401|     provicional|provicional|2026-07 BCO ALFIN|   2026-07-27| BASE BOT|            1|       1|         VDAD|Outbound Auto Dial|           NA|
|10028870|  1|951765150|     0|        10028870|    2026-07-27|           401|     provicional|provicional|2026-07 BCO A

In [56]:
df_subir_01=df_subir_01.withColumnRenamed('dni_cliente','Dni')


In [57]:
df_split=df_split.join(df_subir_01.select('Dni','PESO1'),['Dni'],'left')

In [58]:
df_split=df_split.withColumn('Codigo_Paleta',when(F.col('PESO1').isNotNull(),'zzz2').otherwise(F.col('Codigo_Paleta')))


In [ ]:
df_split=df_split.withColumn('Codigo_Paleta',when(F.col('estado').isNotNull(),'zzz1').otherwise(F.col('Codigo_Paleta')))

In [59]:
df_split = df_split.withColumn(
    "segundos",
    F.when(
        F.col("Codigo_Paleta").isin("PDROP", "AB", "NA", "zzz27", "zzz26", "zzz25",'zzz24','zzz23'),
        F.lit(0)
    )
    .when(
        F.col("Codigo_Paleta") == "zzz2",
        F.floor(F.rand() * 51 + 100).cast("int")   # 100 a 150
    )
    .when(
        F.col("Codigo_Paleta").isin("zzz5", "zzz6", "zzz9", "zzz13", "zzz10", "zzz11"),
        F.floor(F.rand() * 51 + 50).cast("int")    # 30 a 80
    )
    .when(
        F.col("Codigo_Paleta").isin("zzz19", 'zzz15'),
        F.floor(F.rand() * 51 + 30).cast("int")    # 30 a 80
    )
    .otherwise(F.lit(0))
)
codigos_vdad = ["PDROP", "NA", "AB"]


## mas atento

In [62]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

fecha_col = "Fecha_Llamada"

inicio = 16 * 3600          # 16:00
fin = 19 * 3600 + 50 * 60   # 19:50
rango = fin - inicio

df_split = df_split.withColumn("_id", F.monotonically_increasing_id())

# Duración
df_split = df_split.withColumn(
    "_duracion",
    F.coalesce(F.col("segundos").cast("int"), F.lit(0))
)

# =====================================================
# 1. VDAD
# =====================================================
df_vdad = (
    df_split
    .filter(F.col("DNI_Ejecutivo") == "VDAD")
    .withColumn(
        "_seg_inicio",
        F.floor(F.rand() * rango + inicio).cast("int")
    )
)

# =====================================================
# 2. BASE REGULAR (1 llamada por agente)
# =====================================================
# df_regular = df_split.filter(
#     (F.col("TIPO_LOTE") == "BASE REGULAR") &
#     (F.col("DNI_Ejecutivo") != "VDAD")
# )

# w_reg = Window.partitionBy(
#     fecha_col,
#     "DNI_Ejecutivo"
# ).orderBy(F.rand())

# df_regular = df_regular.withColumn(
#     "_rn",
#     F.row_number().over(w_reg)
# )

# df_regular = df_regular.withColumn(
#     "_gap",
#     F.floor(F.rand() * 16 + 5).cast("int")
# )

# w_reg_acum = (
#     Window.partitionBy(fecha_col, "DNI_Ejecutivo")
#     .orderBy("_rn")
#     .rowsBetween(Window.unboundedPreceding, -1)
# )

# df_regular = df_regular.withColumn(
#     "_seg_inicio",
#     inicio +
#     F.coalesce(
#         F.sum(
#             F.col("_duracion") + F.col("_gap")
#         ).over(w_reg_acum),
#         F.lit(0)
#     )
# )

# =====================================================
# 3. BASE BOT (15 llamadas simultáneas)
# =====================================================
df_bot = df_split.filter(
    (F.col("TIPO_LOTE") == "BASE BOT") &
    (F.col("DNI_Ejecutivo") != "VDAD")
)

w_bot = Window.partitionBy(
    fecha_col,
    "DNI_Ejecutivo"
).orderBy(F.rand())

df_bot = df_bot.withColumn(
    "_rn",
    F.row_number().over(w_bot)
)

df_bot = df_bot.withColumn(
    "_grupo_bot",
    F.floor((F.col("_rn") - 1) / 15).cast("int")
)

df_bot_grupos = (
    df_bot
    .groupBy(
        fecha_col,
        "DNI_Ejecutivo",
        "_grupo_bot"
    )
    .agg(
        F.max("_duracion").alias("_duracion_grupo")
    )
)

df_bot_grupos = df_bot_grupos.withColumn(
    "_gap_grupo",
    F.floor(F.rand() * 16 + 5).cast("int")
)

w_bot_acum = (
    Window.partitionBy(fecha_col, "DNI_Ejecutivo")
    .orderBy("_grupo_bot")
    .rowsBetween(Window.unboundedPreceding, -1)
)

df_bot_grupos = df_bot_grupos.withColumn(
    "_seg_inicio",
    inicio +
    F.coalesce(
        F.sum(
            F.col("_duracion_grupo") +
            F.col("_gap_grupo")
        ).over(w_bot_acum),
        F.lit(0)
    )
)

df_bot = df_bot.join(
    df_bot_grupos.select(
        fecha_col,
        "DNI_Ejecutivo",
        "_grupo_bot",
        "_seg_inicio"
    ),
    on=[
        fecha_col,
        "DNI_Ejecutivo",
        "_grupo_bot"
    ],
    how="left"
)



In [63]:
# =====================================================
# UNIR TODO
# =====================================================
df_split = (
    df_bot
    .unionByName(df_vdad, allowMissingColumns=True)
)

# df_split = (
#     df_regular
#     .unionByName(df_bot, allowMissingColumns=True)
#     .unionByName(df_vdad, allowMissingColumns=True)
# )
# Limitar horario máximo
df_split = df_split.withColumn(
    "_seg_inicio",
    F.when(
        F.col("_seg_inicio") > fin,
        F.lit(fin)
    ).otherwise(
        F.col("_seg_inicio").cast("int")
    )
)

# Hora HH:mm:ss
df_split = df_split.withColumn(
    "hora",
    F.date_format(
        F.from_unixtime("_seg_inicio"),
        "HH:mm:ss"
    )
)

# Eliminar columnas auxiliares
df_split = df_split.drop(
    "_id",
    "_duracion",
    "_rn",
    "_gap",
    "_grupo_bot",
    "_seg_inicio",
    "_duracion_grupo",
    "_gap_grupo"
)

In [60]:
from pyspark.sql.window import Window

fecha_col = "Fecha_Llamada"  # cambia esto si tu columna se llama fecha_envio, Fecha_Llamada, etc.

inicio = 16 * 3600          # 09:00:00
# fin = 15 * 3600 + 50 * 60  # 17:50:00
fin = 19 * 3600 + 50 * 60  # 17:50:00
rango = fin - inicio

df_split = df_split.withColumn("_id", F.monotonically_increasing_id())

# Duración de llamada
df_split = df_split.withColumn(
    "_duracion",
    F.when(F.col("segundos").isNull(), F.lit(0))
     .otherwise(F.col("segundos").cast("int"))
)


In [61]:

# =========================
# 1. VDAD: horario libre
# =========================
df_vdad = df_split.filter(F.col("DNI_Ejecutivo") == "VDAD") \
    .withColumn(
        "_seg_inicio",
        F.floor(F.rand() * rango + inicio).cast("int")
    )


In [ ]:

# # =========================
# # 2. BASE REGULAR: sin cruce por agente y fecha
# # =========================
# df_regular = df_split.filter(
#     (F.col("TIPO_LOTE") == "BASE REGULAR") &
#     (F.col("DNI_Ejecutivo") != "VDAD")
# )

# w_reg = Window.partitionBy(fecha_col, "DNI_Ejecutivo").orderBy(F.rand())

# df_regular = df_regular.withColumn("_rn", F.row_number().over(w_reg))

# # pequeño espacio aleatorio entre llamadas: 5 a 20 segundos
# df_regular = df_regular.withColumn(
#     "_gap",
#     F.floor(F.rand() * 16 + 5).cast("int")
# )

# w_reg_acum = Window.partitionBy(fecha_col, "DNI_Ejecutivo") \
#     .orderBy("_rn") \
#     .rowsBetween(Window.unboundedPreceding, -1)

# df_regular = df_regular.withColumn(
#     "_seg_inicio",
#     inicio + F.coalesce(
#         F.sum(F.col("_duracion") + F.col("_gap")).over(w_reg_acum),
#         F.lit(0)
#     )
# )
# =========================
# 3. BASE BOT: hasta 15 llamadas al mismo tiempo
# =========================
df_bot = df_split.filter(
    (F.col("TIPO_LOTE") == "BASE REGULAR") &
    (F.col("DNI_Ejecutivo") != "VDAD")
)

w_bot = Window.partitionBy(fecha_col, "DNI_Ejecutivo").orderBy(F.rand())

df_bot = df_bot.withColumn("_rn", F.row_number().over(w_bot))

# cada grupo de 15 llamadas comparte la misma hora
df_bot = df_bot.withColumn(
    "_grupo_bot",
    F.floor((F.col("_rn") - 1) / 15).cast("int")
)

df_bot_grupos = df_bot.groupBy(fecha_col, "DNI_Ejecutivo", "_grupo_bot") \
    .agg(F.max("_duracion").alias("_duracion_grupo"))

df_bot_grupos = df_bot_grupos.withColumn(
    "_gap_grupo",
    F.floor(F.rand() * 16 + 5).cast("int")
)

w_bot_acum = Window.partitionBy(fecha_col, "DNI_Ejecutivo") \
    .orderBy("_grupo_bot") \
    .rowsBetween(Window.unboundedPreceding, -1)

df_bot_grupos = df_bot_grupos.withColumn(
    "_seg_inicio",
    inicio + F.coalesce(
        F.sum(F.col("_duracion_grupo") + F.col("_gap_grupo")).over(w_bot_acum),
        F.lit(0)
    )
)


In [55]:

df_bot = df_bot.join(
    df_bot_grupos.select(fecha_col, "DNI_Ejecutivo", "_grupo_bot", "_seg_inicio"),
    on=[fecha_col, "DNI_Ejecutivo", "_grupo_bot"],
    how="left"
)

In [56]:


# =========================
# Unir todo
# =========================
df_split = df_regular.unionByName(df_bot, allowMissingColumns=True) \
                     .unionByName(df_vdad, allowMissingColumns=True)

# Si se pasa de 17:50, lo limitamos a 17:50
df_split = df_split.withColumn(
    "_seg_inicio",
    F.when(F.col("_seg_inicio") > fin, F.lit(fin))
     .otherwise(F.col("_seg_inicio").cast("int"))
)

# Crear hora formato HH:mm:ss
df_split = df_split.withColumn(
    "hora",
    F.date_format(
        F.from_unixtime(F.col("_seg_inicio")),
        "HH:mm:ss"
    )
)

# Limpiar columnas auxiliares
df_split = df_split.drop(
    "_id", "_duracion", "_rn", "_gap",
    "_grupo_bot", "_seg_inicio",
    "_duracion_grupo", "_gap_grupo"
)

In [ ]:
xdxdxdxdxdxdxdxdxdxdxdxd

In [64]:
from pyspark.sql import functions as F

df_split = df_split.withColumn(
    "Fecha_Hora_Llamada",
    F.to_timestamp(
        F.concat_ws(
            " ",
            F.date_format(F.col("Fecha_Llamada"), "yyyy-MM-dd"),
            F.col("hora")
        ),
        "yyyy-MM-dd HH:mm:ss"
    )
)

df_split = df_split.withColumn(
    "Inicio",
    F.col("Fecha_Hora_Llamada")
)

df_split = df_split.withColumn(
    "Fin",
    F.from_unixtime(
        F.unix_timestamp(F.col("Inicio")) + F.col("segundos").cast("int")
    ).cast("timestamp")
)

df_split = df_split.withColumn(
    "Trama_Hora",
    F.hour(F.col("Inicio"))
)

In [65]:

df_split = df_split.withColumn(
    "Trama_Hora",
    F.hour(F.col("Inicio"))
)

In [66]:
df_split=df_split.dropDuplicates(['Dni'])

In [67]:
df_split = df_split.withColumn(
    "Codigo_Paleta",
    F.col("Codigo_Paleta").cast("string")
)

In [61]:
df_split.count()

13591

In [69]:
df_split.join(df_ref,['dni'],'leftanti').count()

0

In [68]:
df_ref.count()

12505

In [71]:
print(df_split.columns)

['Fecha_Llamada', 'DNI_Ejecutivo', 'Dni', '_c0', 'celular', 'retiro', 'NUMERO_DOCUMENTO', 'fecha_registro', 'Numero_Campana', 'list_description', 'list_name', 'Nombre_Campana', 'TIPO_LOTE', 'PROPENSION_IC', 'FRESCURA', 'Ejecutivo', 'Codigo_Paleta', 'PESO1', 'segundos', 'hora', 'Fecha_Hora_Llamada', 'Inicio', 'Fin', 'Trama_Hora']


In [74]:
df_split=df_split.withColumnRenamed(
'celular','PHONE_NUMBER'
)
    

In [75]:
df_split.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description').show(3)


+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+
|     Dni|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|Numero_Campana|Trama_Hora|  list_name|list_description|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+
|10001066|         VDAD|Outbound Auto Dial|2026-07-27 13:39:14|       0|   2026-07-27|        13|   992294104|        PDROP|2026-07-27 13:39:14|2026-07-27 13:39:14|           401|        13|provicional|     provicional|
|10003689|         VDAD|Outbound Auto Dial|2026-07-27 11:14:45|       0|   2026-07-27|        11|   985079299|          

In [65]:
df_split.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'Trama_Hora', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description').show(3)

+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+
|     Dni|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|Numero_Campana|Trama_Hora|  list_name|list_description|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+--------------+----------+-----------+----------------+
|00006534|         VDAD|Outbound Auto Dial|2026-07-01 05:03:35|       0|   2026-07-01|         5|   961091285|        PDROP|2026-07-01 05:03:35|2026-07-01 05:03:35|           401|         5|provicional|     provicional|
|00007374|         VDAD|Outbound Auto Dial|2026-07-01 07:14:39|       0|   2026-07-01|         7|   944408041|        PD

In [ ]:
# df_split=df_split.filter(F.col('Ejecutivo')=='Agente BCO Alfin')

In [76]:
df_split=df_split.withColumn('Sub_estado',F.lit(''))
df_split=df_split.withColumn('Estados',F.lit(''))

In [77]:
df_split=df_split.select('Dni', 'DNI_Ejecutivo', 'Ejecutivo', 'Fecha_Hora_Llamada', 'segundos', 'Fecha_Llamada', 'PHONE_NUMBER', 'Codigo_Paleta', 'Inicio', 'Fin','Numero_Campana','Trama_Hora','list_name','list_description','Estados','Sub_estado')

In [78]:
df_split=df_split.withColumn('peso_ref',F.lit(1))

In [36]:
query = """
    select * ,0 as peso_ref from SAMANTHA.dbo.tmp_llamadas_mes_borrar1
    """
df_ref_vici=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [37]:
print(df_split.count())
print(df_ref_vici.count())

23102
17465


In [79]:
df_split.count()

2782

In [82]:
print(df_split.printSchema())

root
 |-- Dni: string (nullable = true)
 |-- DNI_Ejecutivo: string (nullable = true)
 |-- Ejecutivo: string (nullable = true)
 |-- Fecha_Hora_Llamada: timestamp (nullable = false)
 |-- segundos: integer (nullable = true)
 |-- Fecha_Llamada: string (nullable = false)
 |-- PHONE_NUMBER: string (nullable = true)
 |-- Codigo_Paleta: string (nullable = true)
 |-- Inicio: timestamp (nullable = false)
 |-- Fin: timestamp (nullable = true)
 |-- Numero_Campana: string (nullable = false)
 |-- Trama_Hora: integer (nullable = false)
 |-- list_name: string (nullable = false)
 |-- list_description: string (nullable = false)
 |-- Estados: string (nullable = false)
 |-- Sub_estado: string (nullable = false)
 |-- peso_ref: integer (nullable = false)

None


In [83]:
df_split=df_split.drop('peso_ref')

In [84]:
append_table_SQL(spark,df_split,f'tmp_llamadas_mes',server_zeus,user_zeus,pwd_zeus,'SAMANTHA')


In [210]:
print([row['estado' ] for row in df_split.select('estado').distinct().collect()])


['ENVIADO   ', None]


In [38]:
df_necesito.show(2)
# df_necesito=df_necesito.withColumn('Numero_Campana',F.lit('401'))
# df_necesito=df_necesito.withColumn('list_description',F.lit('provicional'))
# df_necesito=df_necesito.withColumn('list_name',F.lit('provicional'))
# df_necesito=df_necesito.withColumn('Nombre_Campana',F.lit('BOT_ALFIN'))
# df_necesito=df_necesito.withColumn('Nombre_Campana',F.lit('2026-07 BCO ALFIN'))


+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
|     Dni|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|PHONE_NUMBER|Codigo_Paleta|             Inicio|                Fin|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
|72048296|         VDAD|Outbound Auto Dial|2023-11-24 17:34:47|       0|   2023-11-24|        17|   922488467|           AB|2023-11-24 17:34:47|2023-11-24 17:34:47|
|29690194|         VDAD|Outbound Auto Dial|2023-11-24 17:34:47|       0|   2023-11-24|        17|   983941534|           AB|2023-11-24 17:34:47|2023-11-24 17:34:47|
+--------+-------------+------------------+-------------------+--------+-------------+----------+------------+-------------+-------------------+-------------------+
only showi

In [205]:
df_split.show(5)

+--------+-------------+--------+--------------------+---------+----+------------+------------+-----------+-----------+-------------------+--------------+------+-------+--------------+----------------+-----------+-----------------+-------------+-------------+------------------+-------------+
|     Dni|PROPENSION_IC|FRESCURA|             USER_V3|TIPO_LOTE|lote|PHONE_NUMBER|fecha_visita|hora_visita|fecha_envio|intentos_realizados|fecha_registro|estado|por_dia|Numero_Campana|list_description|  list_name|   Nombre_Campana|Fecha_Llamada|DNI_Ejecutivo|         Ejecutivo|Codigo_Paleta|
+--------+-------------+--------+--------------------+---------+----+------------+------------+-----------+-----------+-------------------+--------------+------+-------+--------------+----------------+-----------+-----------------+-------------+-------------+------------------+-------------+
|47319495|            1|       0|2. sunedu & sunarp B| BASE BOT| BOT|   954821826|        NULL|       NULL|       NULL|  

In [ ]:


Bot Bco Alfin



In [186]:

query = """
select top(10)* From [THOTH].dbo.Tmp_LLamadas_Alfin_bot
    """
df_llamadas_bot=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_llamadas_bot.show()

+--------+--------------+--------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+----------------+----------------+------------+------------+-----------+-------------+-------------------+-------------------+--------+--------+--------------------+--------------------+-----+------------------+----------+------------+---+-------+
|     Dni|Numero_Campana|Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|         Descripcion|list_description|       list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin| lead_id| Estado_|         Sub_Estado_|        Descripcion_|Pesos|            Enlace|Fecha_Llam|Hora_Llamada| RH|COD_BCO|
+--------+--------------+--------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+--------------------+---

In [187]:
print([row['DNI_Ejecutivo' ] for row in df_llamadas_bot.select('DNI_Ejecutivo').distinct().collect()])


['49587612', 'VDAD']


In [101]:
print(41539-41668)

-129


In [ ]:
nocet 41668
1760 odo
bot 43428

In [58]:
49587612

['ACTIVO', 'RETIVO_CORREO']


In [2]:
print(86857*.50,'bot')
print((102599+2779)*.3,'hum')
print(192235)

43428.5 bot
31613.399999999998 hum
192235


In [23]:
print([row['estado' ] for row in df_crm.select('estado').distinct().collect()])
# print([row['codigo_ejecutivo_id' ] for row in df_crm.select('codigo_ejecutivo_id').distinct().collect()])


['PENDIENTE ', 'ERROR     ', 'ENVIADO   ']


In [ ]:
['NUMERO_DOCUMENTO', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'Agencia_comercial', 'Region_comercial', 'color_final', 'GRUPO_TASA', 'GRUPO_MONTO', 'tipo_cliente_riegos', 'USER_V3', 'TIPO_CLIENTE', 'FRESCURA', 'TIPO_BASE', 'campania', 'FLG_AAHH', 'INTENSIDAD_MAX', 'REGION', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_TASA', 'TIPO_LOTE', 'TIPO_TELF', 'lote', 'tMontoDesemFugas']prop

['NUMERO_DOCUMENTO', 'NRG', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'TIPO_GESTION_hum', 'GESTION_hum', 'SUBGESTION_hum', 'TIPO_GESTION_bot', 'GESTION_bot', 'SUBGESTION_bot', 'TIPO', 'RECORRIDO', 'RECORRIDO_hum', 'RECORRIDO_bot', 'CET', 'CET_hum', 'CET_bot', 'CONT_GEN', 'CONT_GEN_hum', 'CONT_GEN_bot', 'Hora_Llamada', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Ejecutivo', 'RHFC', 'SUPERVISOR', 'segundos', 'AGENDADOS', 'AGENDADOS_hum', 'AGENDADOS_bot', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'NUMXDNI', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'tMontoDesem', 'CNT_LLAMADAS', 'CNT_LLAMADAS_hum', 'CNT_LLAMADAS_bot', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO', 'RETIRO', 'REP1', 'REP2', 'FLG_SIN_ENR', 'tMesGestion', 'DESCRIPCION', 'DESCRIPCION2', 'llave', 'DEPARTAMENTO', 'DISTRITO', 'PROPENSION_IC', 'CUOTA', 'Agencia_comercial', 'Region_comercial', 'color_final', 'GRUPO_TASA', 'GRUPO_MONTO', '

In [15]:
query = """
    SELECT * FROM maeba.[ADM_OBJ_TG].[tGestionMesAlfin]
    where RECORRIDO = 0
    """
df_recorrido=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_recorrido.count()

156930

In [6]:
df_formato.join(df_recorrido,['NUMERO_DOCUMENTO'],'inner').count()

0

In [29]:
df_recorrido.show()

+----------------+---+----+------------+----------+----------+----------------+-----------+--------------+----------------+-----------+--------------+----------+---------+-------------+-------------+---+-------+-------+--------+------------+------------+------------+--------------+--------+-------------+----+---------+----+-----------+--------+---------+-------------+-------------+--------+--------+--------+--------+---------+-------+------+---------+---------+-------+-------------+----------+-----------+------------+----------------+----------------+-----------------+-----------------+-----------+--------+------+-----+-----+-----------+-----------+--------------------+------------------+-------------------+------------+--------------------+-------------+-------+--------------------+--------------------+---------------+-------------+---------------+-------------------+--------------------+-------------+--------+----------------+--------+------------+--------------+---------------+-----

In [31]:
df_recorrido.count()

1999

In [33]:
# print([row['FEC_NAC' ] for row in df_list.select('FEC_NAC').distinct().collect()])
df_recorrido.groupBy('PROPENSION_IC') \
    .count() \
    .orderBy('PROPENSION_IC') \
    .show(30)


+-------------+-----+
|PROPENSION_IC|count|
+-------------+-----+
|            1|  572|
|            2|  980|
|            3|  132|
|            4|  122|
|            5|  116|
|            6|   77|
+-------------+-----+



In [ ]:
df_recorrido

In [25]:
query = """
    select top(10)* From SAMANTHA..tmp_llamadas_mes
    """
df_recorrido=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
df_recorrido.show(2)

+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|     Dni|Numero_Campana|      Nombre_Campana|DNI_Ejecutivo|         Ejecutivo| Fecha_Hora_Llamada|segundos|Fecha_Llamada|Trama_Hora|Estados|Sub_estado|Descripcion|    list_description|           list_name|PHONE_NUMBER|Fecha_Agenda|Comentarios|Codigo_Paleta|             Inicio|                Fin|lead_id|
+--------+--------------+--------------------+-------------+------------------+-------------------+--------+-------------+----------+-------+----------+-----------+--------------------+--------------------+------------+------------+-----------+-------------+-------------------+-------------------+-------+
|72048296|            80|2023-11 CENCOSUD ...|         VDAD|Outbound Auto Dial|

In [ ]:
RetiroDefinitivo_BlackList